# 0. Imports

## 0.1 Packages

In [1]:
import asyncio
import nest_asyncio
from tenacity import retry, wait_exponential, stop_after_attempt
import aiohttp
import pandas as pd

## 0.2 Data

In [2]:
ObsList = pd.read_csv(r"../Data Raw/ObsList.csv", sep=";")

# 1. Taxonomy Uniformization

## 1.1. Code

### 1.1.1. GBIF: Names Check

#### 1.1.1.1. Uniformization

In [3]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Species_1(session, species):

    url = f"https://api.gbif.org/v1/species/match?name={species}"  
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('rank') == 'SPECIES':
                if data.get('status') != 'ACCEPTED':
                    accepted_name = data.get('species') or data.get('canonicalName')
                else:
                    accepted_name = data.get('canonicalName')
                
                return accepted_name if accepted_name else None
            else:
                return species
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def GBIF_Sessions_1(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Species_1(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def GBIF_Extractor_1(species_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Sessions_1(species_list))

#### 1.1.1.2. Filter

In [4]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Species_2(session, species):

    url = f"https://api.gbif.org/v1/species/match?name={species}"  
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('rank') == 'SPECIES':
                if data.get('status') != 'ACCEPTED':
                    accepted_name = data.get('species') or data.get('canonicalName')
                else:
                    accepted_name = data.get('canonicalName')
                
                return accepted_name if accepted_name else None
            else:
                return None
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def GBIF_Sessions_2(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Species_2(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def GBIF_Extractor_2(species_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Sessions_2(species_list))

### 1.1.2. Global Names Verifier: Cross-check

#### 1.1.2.1. Uniformization

In [5]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def VNF_Species_1(session, species):
    
    url = f"https://verifier.globalnames.org/api/v1/verifications/{species}?data_sources=1%7C12&all_matches=false&capitalize=false&species_group=false&fuzzy_uninomial=false&stats=true&main_taxon_threshold=0.5"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get("names", [{}])[0].get("bestResult", {}).get("taxonomicStatus") == 'Accepted':
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('matchedCanonicalSimple')
            else:
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('currentCanonicalSimple')
            return accepted_name if accepted_name != "" else species
            
            
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def VNF_Sessions_1(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [VNF_Species_1(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def VNF_Extractor_1(species_list):
    return asyncio.get_event_loop().run_until_complete(VNF_Sessions_1(species_list))

#### 1.1.2.2. Filter

In [6]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def VNF_Species_2(session, species):
    
    url = f"https://verifier.globalnames.org/api/v1/verifications/{species}?data_sources=1%7C12&all_matches=false&capitalize=false&species_group=false&fuzzy_uninomial=false&stats=true&main_taxon_threshold=0.5"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get("names", [{}])[0].get("bestResult", {}).get("taxonomicStatus") == 'Accepted':
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('matchedCanonicalSimple')
            else:
                accepted_name = data.get("names", [{}])[0].get("bestResult", {}).get('currentCanonicalSimple')
            return accepted_name
            
            
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def VNF_Sessions_2(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [VNF_Species_2(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def VNF_Extractor_2(species_list):
    return asyncio.get_event_loop().run_until_complete(VNF_Sessions_2(species_list))

### 1.1.3. GBIF: Family Extract

In [7]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Family(session, species):
    url = f"https://api.gbif.org/v1/species/match?name={species}"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('family'):
                return data.get('family')
            else:
                return None
    
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def GBIF_Family_Sessions(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Family(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def GBIF_Family_Extract(species_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Family_Sessions(species_list))

### 1.1.4. GBIF: Lepidoptera Cross-check

In [8]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Lepidoptera(session, family):
    
    url = f"https://api.gbif.org/v1/species/match?name={family}"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('order') == 'Lepidoptera':
                return 1
            elif data.get('order') is None:
                return None
            else:
                return 0
            
    except Exception as e:
        print(f"Error fetching data for {family}: {e}")
        return False

async def GBIF_Lepidoptera_Sessions(family_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Lepidoptera(session, family) for family in family_list]
        return await asyncio.gather(*tasks)

def GBIF_Lepidoptera_Extract(family_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Lepidoptera_Sessions(family_list))

## 1.2. Data

In [9]:
ObsList['Species'] = ObsList['Species'].apply(lambda x: ' '.join(x.split()[:2]))

In [10]:
ObsList['AcceptedSpecies'] = GBIF_Extractor_2(VNF_Extractor_1(GBIF_Extractor_1(ObsList['Species'])))
ObsList['AcceptedSpecies'] = ObsList['AcceptedSpecies'].apply(lambda x: ' '.join(x.split()[:2]) if isinstance(x, str) else x)

In [11]:
ObsList[ObsList['AcceptedSpecies'].isna()]['Species'].unique()

array(['Heliocheilus cystiphora', 'Rheumaptera affirmata',
       'Spodoptera sunia', 'Heliocontia margana', 'Trissodoris guamensis',
       'Prospalta dolorosa', 'Phalaenophana fadusalis',
       'Opodiphthera eucalypti', 'Euphaedra temeraria', 'Orgyia basalis'],
      dtype=object)

In [12]:
ObsList[ObsList['AcceptedSpecies'].isna()].shape[0]


13

In [13]:
ObservationsRaw = ObsList.copy()
ObservationsRaw.dropna(subset=['AcceptedSpecies'], inplace=True)
ObservationsRaw[ObservationsRaw['AcceptedSpecies'].isna()]

,Species,NAME_0,Realm,Cryptogenic,Dispersal,Eradicated,Intentional_Release,Introduced,Established,Observation Year,Reference,Reference Year,AcceptedSpecies


In [14]:
ObservationsRaw.reset_index(drop=True, inplace=True)
ObservationsRaw.to_csv(r'../Transformed Data/ObservationsRaw.csv', sep =';')

# 2. Taxonomy Table Extract

In [15]:
Taxonomy = pd.DataFrame()
Taxonomy['Species'] = [*ObservationsRaw['Species'], *ObservationsRaw['AcceptedSpecies']]
Taxonomy.drop_duplicates(subset=['Species'], inplace=True)
Taxonomy.reset_index(drop=True, inplace=True)

## 2.1 Accepted Species

In [16]:
Taxonomy['AcceptedSpecies'] = GBIF_Extractor_2(VNF_Extractor_1(GBIF_Extractor_1(Taxonomy['Species'])))

## 2.2. Genus

In [17]:
Taxonomy['Genus'] = Taxonomy['AcceptedSpecies'].astype(str).str.split().str[0]


## 2.3. Family

In [20]:
Taxonomy['Family'] = GBIF_Family_Extract(Taxonomy['AcceptedSpecies'])
Taxonomy.reset_index(drop=True, inplace=True)

In [21]:
Taxonomy[Taxonomy['Family'].isna()]

,Species,AcceptedSpecies,Genus,Family
904,Homoeographa lamceolella,Homoeographa lanceolella,Homoeographa,None
1019,Calephelis virginiensis,Calephelis virginiensis,Calephelis,None
1591,Homoeographa lanceolella,Homoeographa lanceolella,Homoeographa,None


## 2.4. Lepidoptera Cross-Check

In [22]:
Taxonomy['Lepidoptera'] = GBIF_Lepidoptera_Extract(Taxonomy['Family'])
Taxonomy.reset_index(drop=True, inplace=True)

In [23]:
Taxonomy[(Taxonomy['Lepidoptera'].isna()) | (Taxonomy['Lepidoptera'] == 0)]['Family'].unique()

array(['Saturniidae', None], dtype=object)

## 2.5 Manual Updates

In [24]:
Taxonomy.loc[Taxonomy['AcceptedSpecies'] == 'Homoeographa lanceolella', 'Family'] = 'Pyralidae'
Taxonomy.loc[Taxonomy['AcceptedSpecies'] == 'Calephelis virginiensis', 'Family'] = 'Lycaenidae'

In [25]:
Butterfly_Families = ["Papilionidae", "Hedylidae", "Hesperiidae", "Pieridae", "Lycaenidae", "Riodinidae", "Nymphalidae"]

Moth_Families = ["Micropterigidae", "Agathiphagidae", "Heterobathmiidae", "Eriocraniidae", "Aenigmatineidae", "Acanthopteroctetidae", 
                      "Lophocoronidae", "Neopseustidae", "Mnesarchaeidae", "Hepialidae", "Nepticulidae", "Opostegidae", "Andesianidae",
                      "Heliozelidae", "Adelidae", "Incurvariidae", "Cecidosidae", "Prodoxidae", "Tridentaformidae", "Palaephatidae", 
                      "Tischeriidae", "Millieriidae", "Eriocottidae", "Psychidae", "Tineidae", "Meessiidae", "Dryadaulidae", "Roeslerstammiidae",
                      "Bucculatricidae", "Gracillariidae", "Yponomeutidae", "Ypsolophidae", "Plutellidae", "Glyphipterigidae", "Argyresthiidae", 
                      "Lyonetiidae", "Attevidae", "Praydidae", "Heliodinidae", "Bedelliidae", "Scythropiidae", "Douglasiidae", "Simaethistidae", 
                      "Autostichidae", "Lecithoceridae", "Xyloryctidae", "Oecophoridae", "Depressariidae", "Cosmopterigidae", "Gelechiidae", 
                      "Elachistidae", "Coleophoridae", "Batrachedridae", "Scythrididae", "Blastobasidae", "Stathmopodidae", "Momphidae", "Pterolonchidae",
                      "Lypusidae", "Schistonoeidae", "Tineodidae", "Alucitidae", "Pterophoridae", "Copromorphidae", "Carposinidae", "Schreckensteiniidae", 
                      "Epermeniidae", "Urodidae", "Immidae", "Choreutidae", "Galacticidae", "Tortricidae", "Brachodidae", "Cossidae", "Dudgeoneidae", 
                      "Metarbelidae", "Ratardidae", "Castniidae", "Sesiidae", "Epipyropidae", "Cyclotornidae", "Heterogynidae", "Lacturidae", "Phaudidae", 
                      "Dalceridae", "Limacodidae", "Megalopygidae", "Aididae", "Somabrachyidae", "Himantopteridae", "Zygaenidae", "Whalleyanidae", 
                      "Thyrididae", "Hyblaeidae", "Prodidactidae", "Callidulidae", "Pyralidae", "Crambidae", "Mimallonidae", "Cimeliidae", "Doidae", 
                      "Drepanidae", "Lasiocampidae", "Apatelodidae", "Eupterotidae", "Brahmaeidae", "Phiditiidae", "Anthelidae", "Carthaeidae", 
                      "Endromidae", "Bombycidae", "Saturniidae", "Sphingidae", "Epicopeiidae", "Sematuridae", "Uraniidae", "Geometridae", "Pseudobistonidae", 
                      "Oenosandridae", "Notodontidae", "Erebidae", "Euteliidae", "Nolidae", "Noctuidae"]

In [26]:
Taxonomy['Butterfly'] = Taxonomy['Family'].isin(Butterfly_Families).astype(int)

In [27]:
Taxonomy[(Taxonomy['Lepidoptera'] == 0) | (Taxonomy['Lepidoptera'].isna())]['Family'].unique()

array(['Saturniidae', 'Pyralidae', 'Lycaenidae'], dtype=object)

In [28]:
Taxonomy[(Taxonomy['Lepidoptera'] == 0) | (Taxonomy['Lepidoptera'].isna())]

,Species,AcceptedSpecies,Genus,Family,Lepidoptera,Butterfly
179,Antheraea pernyi,Antheraea pernyi,Antheraea,Saturniidae,NaN,0
358,Saturnia japonica,Saturnia japonica,Saturnia,Saturniidae,NaN,0
591,Hyalophora euryalus,Hyalophora euryalus,Hyalophora,Saturniidae,NaN,0
772,Samia cynthia,Samia cynthia,Samia,Saturniidae,NaN,0
904,Homoeographa lamceolella,Homoeographa lanceolella,Homoeographa,Pyralidae,NaN,0
982,Actias selene,Actias selene,Actias,Saturniidae,NaN,0
987,Antheraea yamamai,Antheraea yamamai,Antheraea,Saturniidae,NaN,0
991,Antheraea paphia,Antheraea paphia,Antheraea,Saturniidae,NaN,0
992,Antheraea polyphemus,Antheraea polyphemus,Antheraea,Saturniidae,NaN,0
1019,Calephelis virginiensis,Calephelis virginiensis,Calephelis,Lycaenidae,NaN,1


As: Lycaenidae, Pyralidae and Saturniidae are all Lepidopterans - drop Lepidoptera crosscheck column as it would be a constant.

In [29]:
Taxonomy.drop(columns=['Lepidoptera'], inplace=True)

In [30]:
Taxonomy

,Species,AcceptedSpecies,Genus,Family,Butterfly
0,Ostrinia nubilalis,Ostrinia nubilalis,Ostrinia,Crambidae,0
1,Agrotis ipsilon,Agrotis ipsilon,Agrotis,Noctuidae,0
2,Zeiraphera diniana,Zeiraphera griseana,Zeiraphera,Tortricidae,0
3,Plutella xylostella,Plutella xylostella,Plutella,Plutellidae,0
4,Spodoptera frugiperda,Spodoptera frugiperda,Spodoptera,Noctuidae,0
...,...,...,...,...,...
1626,Microsphecia tineiformis,Microsphecia tineiformis,Microsphecia,Sesiidae,0
1627,Acraea macarista,Acraea macarista,Acraea,Nymphalidae,1
1628,Apisa metarctiodes,Apisa metarctiodes,Apisa,Erebidae,0
1629,Telchinia encedon,Telchinia encedon,Telchinia,Nymphalidae,1


## 2.6 Export

In [31]:
Taxonomy.to_csv(r'../Data Raw/TaxonomyRaw.csv', index=False, sep = ";")